# The Price Is Right — GPT-OSS 120B Full Fine-Tuning + Evaluation
### openai/gpt-oss-120b · MXFP4 · ed-donner/items_prompts_full · Single H100 80 GB

**Why full fine-tuning instead of QLoRA?**
- GPT-OSS 120B is a MoE model: only **5.1B parameters are active** per forward pass
- This means gradients and optimizer states are computed only for activated experts per step — far less memory pressure than a dense 120B model
- Full SFT (no adapters) allows weight updates across all expert layers, capturing richer task-specific representations that LoRA rank-limits
- MXFP4 quantization keeps the full 117B weight tensor in ~64 GB — fitting the H100 for inference
- For training we use **DeepSpeed ZeRO-3 + CPU offload** to spill optimizer states and gradients to host RAM, making full fine-tuning tractable on a single node

**Why DeepSpeed ZeRO-3 over Unsloth here?**
- Unsloth does not support GPT-OSS 120B (MoE + harmony format + MXFP4 stack)
- ZeRO-3 shards optimizer states, gradients, and optionally parameters across available memory tiers (GPU → CPU RAM → NVMe)
- With Adafactor optimizer (no first-moment state) + ZeRO-3 CPU offload, full fine-tuning of a 120B MoE fits on a single H100 node with ≥256 GB host RAM

**Notebook flow**
1. Environment setup — cache redirect, installs, imports, DeepSpeed config
2. Constants & hyperparameters
3. HuggingFace login
4. Load & inspect dataset
5. Load base model in MXFP4
6. Base model evaluation — diverse beam search + geometric mean
7. Prepare dataset — harmony chat format
8. Full fine-tuning with TRL SFTTrainer + DeepSpeed ZeRO-3
9. Save fine-tuned model
10. Load fine-tuned model for inference
11. Fine-tuned model evaluation
12. Side-by-side comparison & results


In [1]:
! pip install -U kernels 

## 1 · Environment Setup

In [2]:
import os


In [3]:
import os
# get huggingface and wandb api key here using os.getenv() and print them out to verify they are set correctly; pass manually if there is an issue

# ── Redirect HF cache to scratch partition — must happen before any HF import ──
os.environ["HF_HOME"]            = "/tmp/mawojide/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/mawojide/hf_cache/transformers"
os.environ["HF_DATASETS_CACHE"]  = "/tmp/mawojide/hf_cache/datasets"
os.environ["HF_TEMP_DIR"]        = "/tmp/huggingface_temp"   # temp upload/download scratch

os.makedirs("/tmp/mawojide/hf_cache/transformers", exist_ok=True)
os.makedirs("/tmp/mawojide/hf_cache/datasets",     exist_ok=True)

print("HF_HOME →", os.environ["HF_HOME"])

# Verify the hub cache is actually on /tmp — not the home directory
from huggingface_hub import constants
print("HF_HUB_CACHE →", constants.HF_HUB_CACHE)   # must show /tmp/mawojide/hf_cache/hub


HF_HOME → /tmp/mawojide/hf_cache
HF_HUB_CACHE → /tmp/mawojide/hf_cache/hub


In [4]:
# Install dependencies
# GPT-OSS 120B requires transformers ≥ 4.47 for MXFP4 support + harmony chat template
# deepspeed ≥ 0.15 for ZeRO-3 CPU offload stability
# openai-harmony provides the harmony tokenizer utilities used by gpt-oss
! pip install "transformers>=4.47" trl deepspeed accelerate datasets huggingface_hub -q
! pip install openai-harmony scikit-learn plotly tqdm -q
! pip install wandb weave   # experiment tracking + Weave tracing



In [5]:
import os, re, json, math, statistics, gc
from datetime import datetime
from itertools import accumulate
from IPython.display import clear_output

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    set_seed,
)
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from huggingface_hub import login, HfApi

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
capability = torch.cuda.get_device_capability()
USE_BF16   = capability[0] >= 8   # True on H100 (sm_90)
print(f"bfloat16       : {USE_BF16}  (compute capability {capability})")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM           : {vram_gb:.1f} GB")


/home/mawojide/orchard_project_trial/orchard_venv_3_11/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


torch          : 2.10.0+cu128
CUDA available : True
GPU            : NVIDIA H100 80GB HBM3
bfloat16       : True  (compute capability (9, 0))
VRAM           : 85.0 GB


In [6]:
# ── Confirm clean GPU state before anything is loaded ──────────────────────
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM reserved  : {torch.cuda.memory_reserved()  / 1e9:.2f} GB")


VRAM allocated : 0.00 GB
VRAM reserved  : 0.00 GB


In [7]:
# ── Write DeepSpeed ZeRO-3 config to disk ───────────────────────────────────
# ZeRO Stage 3  : shards model params + gradients + optimizer states across all memory tiers
# offload_optimizer → cpu : spills Adam/Adafactor states to host RAM
# overlap_comm          : overlaps gradient reduction with backward pass
# contiguous_gradients  : reduces gradient fragmentation
# reduce_bucket_size    : tuned for H100 NVLink bandwidth
# stage3_param_persistence_threshold: keep small params on GPU to reduce comms overhead

DS_CONFIG_PATH = "/tmp/mawojide/ds_zero3.json"

ds_config = {
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param":     {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": 5e8,
        "stage3_prefetch_bucket_size": 5e8,
        "stage3_param_persistence_threshold": 1e6,
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "steps_per_print": 10,
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "bf16": {"enabled": True},
    "fp16": {"enabled": False},
    "zero_allow_untested_optimizer": True
}

os.makedirs("/tmp/mawojide", exist_ok=True)
with open(DS_CONFIG_PATH, "w") as f:
    json.dump(ds_config, f, indent=2)

print(f"DeepSpeed config written → {DS_CONFIG_PATH}")


DeepSpeed config written → /tmp/mawojide/ds_zero3.json


## 2 · Constants & Hyperparameters

In [8]:
# ── Model ──────────────────────────────────────────────────────────────────
# BASE_MODEL   = "openai/gpt-oss-120b"
BASE_MODEL   = "openai/gpt-oss-20b"
PROJECT_NAME = "price"
HF_USER      = "martinsawojide"

# ── Dataset ────────────────────────────────────────────────────────────────
LITE_MODE         = True
DATA_USER         = "ed-donner"
DATASET_NAME      = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# Load dataset early — sizes needed for VAL_SIZE / EVAL_SIZE below
dataset = load_dataset(DATASET_NAME)
train   = dataset["train"]
val     = dataset["val"]
test    = dataset["test"]
print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")

# ── Run naming ─────────────────────────────────────────────────────────────
RUN_NAME         = f"{datetime.now():%Y-%m-%d_%H.%M.%S}" + ("-lite" if LITE_MODE else "")
# PROJECT_RUN_NAME = f"{PROJECT_NAME}-gpt-oss-120b-{RUN_NAME}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-gpt-oss-20b-{RUN_NAME}"
HUB_MODEL_NAME   = f"{HF_USER}/{PROJECT_RUN_NAME}"
CHECKPOINT_DIR   = f"/tmp/checkpoint_dir/{PROJECT_RUN_NAME}"

# ── Sequence length ────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 512

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS          = 1   if LITE_MODE else 3
BATCH_SIZE      = 1
GRAD_ACCUM      = 8   if LITE_MODE else 16
LEARNING_RATE   = 2e-5
WARMUP_STEPS    = 20  if LITE_MODE else 50
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.01
# OPTIMIZER       = "adafactor"
OPTIMIZER = "adamw_torch"
MAX_GRAD_NORM   = 1.0

# ── Logging / saving ───────────────────────────────────────────────────────
LOG_STEPS    = 5    if LITE_MODE else 10
SAVE_STEPS   = 50   if LITE_MODE else 100
VAL_SIZE     = len(val)    # use full val set — sizes known from dataset load above
LOG_TO_WANDB = True

if LOG_TO_WANDB:
    import wandb
    os.environ["WANDB_API_KEY"]   = os.getenv("WANDB_API_KEY", "")
    os.environ["WANDB_PROJECT"]   = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "false"
    os.environ["WANDB_WATCH"]     = "false"
    wandb.login()

# ── Inference ──────────────────────────────────────────────────────────────
# Updated beam config validated on the Qwen run: 10 groups + 1.5 penalty
# outperforms the original 5 groups + 1.0 penalty configuration.
BEAM_NUM_BEAMS   = 10
BEAM_NUM_GROUPS  = 10     # updated from 5 — validated improvement
BEAM_DIV_PENALTY = 1.5   # updated from 1.0 — higher diversity across beams
EVAL_SIZE        = len(test)   # evaluate on full test set
# REASONING_EFFORT = "low"
REASONING_EFFORT = "medium"

print(f"Run name       : {RUN_NAME}")
print(f"Hub model      : {HUB_MODEL_NAME}")
print(f"Checkpoint dir : {CHECKPOINT_DIR}")
print(f"Dataset        : {DATASET_NAME}")
print(f"Max seq length : {MAX_SEQ_LENGTH}")
print(f"Batch / accum  : {BATCH_SIZE} / {GRAD_ACCUM}  (effective {BATCH_SIZE * GRAD_ACCUM})")
print(f"Optimizer      : {OPTIMIZER}")
print(f"Epochs         : {EPOCHS}")


README.md:   0%|          | 0.00/509 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/216k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/218k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: awojidemartins (awojidemartins_na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train : 20,000
Val   : 1,000
Test  : 1,000
Run name       : 2026-03-08_01.22.31-lite
Hub model      : martinsawojide/price-gpt-oss-20b-2026-03-08_01.22.31-lite
Checkpoint dir : /tmp/checkpoint_dir/price-gpt-oss-20b-2026-03-08_01.22.31-lite
Dataset        : ed-donner/items_prompts_lite
Max seq length : 512
Batch / accum  : 1 / 8  (effective 8)
Optimizer      : adamw_torch
Epochs         : 1


## 3 · HuggingFace Login

In [9]:
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token, add_to_git_credential=True)

# Verify token role — must be 'write' to push to Hub
api = HfApi()
try:
    user = api.whoami()
    print(f"Logged in as : {user['name']}")
    print(f"Token role   : {user['auth']['accessToken']['role']}")
except Exception as e:
    print(f"Login failed : {e}")


Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Logged in as : martinsawojide
Token role   : write


## 4 · Load & Inspect Dataset

In [10]:
dataset = load_dataset(DATASET_NAME)
train   = dataset["train"]
val     = dataset["val"].select(range(VAL_SIZE))
test    = dataset["test"]

print(f"Train : {len(train):,} rows")
print(f"Val   : {len(val):,} rows")
print(f"Test  : {len(test):,} rows")
print(f"\nColumns : {train.column_names}")
print(f"\nSample prompt (first 300 chars):\n{train[0]['prompt'][:300]}")
print(f"\nSample completion : {train[0]['completion']}")


Train : 20,000 rows
Val   : 1,000 rows
Test  : 1,000 rows

Columns : ['prompt', 'completion']

Sample prompt (first 300 chars):
What does this cost to the nearest dollar?

Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  
Category: Home Hardware  
Brand: Schlage  
Description: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  
Details: Designed for a 4" mi

Sample completion : 64.00


## 5 · Load Base Model in MXFP4

GPT-OSS 120B ships with **MXFP4-quantized MoE weights** baked in — no runtime quantization config required. Loading with `torch_dtype=torch.bfloat16` keeps non-MoE layers (attention, layer norms) in full bf16 while the MoE expert weights remain in their native MXFP4 format.

**Harmony format note:** GPT-OSS was trained exclusively on the harmony response format. The tokenizer's `apply_chat_template` handles this automatically — always use it rather than raw string concatenation.


In [11]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# GPT-OSS tokenizer: pad_token is already set; confirm alignment
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # SFTTrainer expects right-padding for packing

print(f"Vocab size     : {tokenizer.vocab_size:,}")
print(f"EOS token      : '{tokenizer.eos_token}'  (id {tokenizer.eos_token_id})")
print(f"PAD token      : '{tokenizer.pad_token}'  (id {tokenizer.pad_token_id})")
print(f"Chat template  : {'present' if tokenizer.chat_template else 'MISSING — check transformers version'}")


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Vocab size     : 199,998
EOS token      : '<|return|>'  (id 200002)
PAD token      : '<|endoftext|>'  (id 199999)
Chat template  : present


In [12]:
# Load base model for evaluation only — inference dtype, no training overhead.
# For training we reload separately inside the trainer to allow ZeRO-3 sharding.
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype   = torch.bfloat16,   # non-MoE layers in bf16; MoE stays in MXFP4
    device_map    = "cuda",           # single H100 — all layers on GPU
    attn_implementation = "kernels-community/vllm-flash-attn3",
)
base_model.eval()

# Set reasoning effort: gpt-oss supports runtime configuration via generation config
# "low" disables extended chain-of-thought — essential for fast, direct price output
base_model.generation_config.reasoning_effort = REASONING_EFFORT

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded   : {BASE_MODEL}")
print(f"VRAM used      : {vram_used:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model loaded   : openai/gpt-oss-20b
VRAM used      : 13.8 GB / 85 GB


## 6 · Inference Helpers

Two key differences from the Qwen notebook:

1. **Harmony format** — prompts must be wrapped via `tokenizer.apply_chat_template`. Passing raw strings will produce garbage output because the model never saw that format during training.
2. **No Unsloth** — we call `model.generate` directly. `torch.inference_mode()` replaces Unsloth's `for_inference` kernel switch.


In [13]:
def build_harmony_prompt(item: dict, include_completion: bool = False) -> str:
    """
    Wrap a dataset item in GPT-OSS harmony chat format.
    apply_chat_template handles the <|start|>/<|message|>/<|end|> tokens.

    If include_completion=True, appends the price as the assistant turn
    (used when building SFT training sequences).
    If False, leaves the assistant turn open (used for inference).
    """
    messages = [{"role": "user", "content": str(item["prompt"])}]

    if include_completion:
        messages.append({
            "role": "assistant",
            "content": str(item["completion"])
        })
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


# Verify harmony wrapping on a sample
sample_text = build_harmony_prompt(train[0])
print("Harmony-formatted inference prompt:")
print(sample_text[:400])
print("...")
token_count = len(tokenizer.encode(sample_text))
print(f"\nToken count (no completion): {token_count}")


Harmony-formatted inference prompt:
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-03-08

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>What does this cost to the nearest dollar?

Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  
Category: 
...

Token count (no completion): 168


In [14]:
def extract_price(text: str):
    """Extract the first plausible USD price from generated text."""
    patterns = [
        r"\$[\d,]+\.?\d*",   # $12.99  $1,299
        r"[\d,]+\.\d{2}",    # 12.99
        r"[\d,]+",            # 12
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            price_str = match.group().replace("$", "").replace(",", "")
            try:
                price = float(price_str)
                if 0.01 <= price <= 100_000:
                    return price
            except ValueError:
                continue
    return None


def predict_with_model(model, tok, item,
                       num_beams=BEAM_NUM_BEAMS,
                       num_groups=BEAM_NUM_GROUPS,
                       diversity_penalty=BEAM_DIV_PENALTY):
    """
    Diverse beam search → geometric mean of all beam prices.
    reasoning_effort="low" keeps the model from generating chain-of-thought
    text before the price, which would consume all 8 max_new_tokens.
    """
    prompt       = build_harmony_prompt(item, include_completion=False)
    inputs       = tok(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens       = 32,           # slightly more than Qwen: harmony adds a few overhead tokens
            max_length           = None,
            num_beams            = num_beams,
            num_beam_groups      = num_groups,
            diversity_penalty    = diversity_penalty,
            num_return_sequences = num_beams,
            pad_token_id         = tok.eos_token_id,
            use_cache            = True,
            trust_remote_code    = True,          # allow loading community attention kernels
            do_sample            = False          # deterministic decoding for evaluation
        )

    prices = []
    for beam_output in output_ids:
        text  = tok.decode(beam_output[input_length:], skip_special_tokens=True)
        price = extract_price(text)
        if price:
            prices.append(price)

    if not prices:
        return None
    return statistics.geometric_mean(prices)


# Named wrapper for Tester.make_title()
def base_model_predict(item):
    return predict_with_model(base_model, tokenizer, item)


## 7 · Evaluation Framework (Tester + Plotly Charts)

In [15]:
GREEN     = "\033[92m"
YELLOW    = "\033[93m"
RED       = "\033[91m"
RESET     = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}


class Tester:
    def __init__(self, predictor, data, title=None, size=EVAL_SIZE):
        self.predictor = predictor
        self.data      = data
        self.title     = title or self.make_title(predictor)
        self.size      = size
        self.titles    = []
        self.guesses   = []
        self.truths    = []
        self.errors    = []
        self.colors    = []

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title()

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        return value if value is not None else 0

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:    return "green"
        elif error < 80 or error / truth < 0.4:  return "orange"
        else:                                     return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value     = self.predictor(datapoint)
        guess     = self.post_process(value)
        truth     = float(datapoint["completion"])
        error     = abs(guess - truth)
        color     = self.color_for(error, truth)
        pieces    = datapoint["prompt"].split("Title: ")
        title     = pieces[1].split("\n")[0] if len(pieces) > 1 else pieces[0]
        title     = title if len(title) <= 40 else title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame({
            "truth": self.truths, "guess": self.guesses,
            "title": self.titles, "error": self.errors, "color": self.colors,
        })
        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]
        max_val = float(max(df["truth"].max(), df["guess"].max()))
        fig = px.scatter(
            df, x="truth", y="guess", color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title, labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=800, height=600,
        )
        for tr in fig.data:
            mask            = df["color"] == tr.name
            tr.customdata   = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val], mode="lines",
            line=dict(width=2, dash="dash", color="deepskyblue"),
            hoverinfo="skip", showlegend=False,
        ))
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n               = len(self.errors)
        running_sums    = list(accumulate(self.errors))
        x               = list(range(1, n + 1))
        running_means   = [s / i for s, i in zip(running_sums, x)]
        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds    = [
            math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
            for i, sq, m in zip(x, running_squares, running_means)
        ]
        ci    = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]
        title = f"{self.title}  |  Final MAE: ${running_means[-1]:,.2f} ± ${ci[-1]:,.2f}"
        fig   = go.Figure()
        fig.add_trace(go.Scatter(
            x=x + x[::-1], y=upper + lower[::-1], fill="toself",
            fillcolor="rgba(128,128,128,0.2)",
            line=dict(color="rgba(255,255,255,0)"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=x, y=running_means, mode="lines",
            line=dict(width=3, color="firebrick"),
            customdata=list(zip(ci)),
            hovertemplate="n=%{x}<br>Avg Error=$%{y:,.2f}<br>±95% CI=$%{customdata[0]:,.2f}<extra></extra>",
        ))
        fig.update_layout(
            title=title, xaxis_title="Number of Datapoints",
            yaxis_title="MAE ($)", width=800, height=300,
            template="plotly_white", showlegend=False,
        )
        fig.show()

    def report(self):
        avg_error = sum(self.errors) / self.size
        mse       = mean_squared_error(self.truths, self.guesses)
        r2        = r2_score(self.truths, self.guesses) * 100
        title     = (f"{self.title}<br>"
                     f"<b>MAE:</b> ${avg_error:,.2f}  "
                     f"<b>MSE:</b> {mse:,.0f}  "
                     f"<b>r²:</b> {r2:.1f}%")
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size), desc=self.title):
            title, guess, truth, error, color = self.run_datapoint(i)
            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        clear_output(wait=True)
        self.report()
        return self


def evaluate(function, data, size=EVAL_SIZE):
    return Tester(function, data, size=size).run()


## 8 · Base Model Evaluation

Benchmark the unmodified GPT-OSS 120B (MXFP4) with diverse beam search + geometric mean.


In [16]:
# Smoke test — single item before committing to full eval
sample = test[0]
pred   = base_model_predict(sample)
truth  = float(sample["completion"])
print(f"Prompt (tail) : ...{sample['prompt'][-80:]}")
print(f"Prediction    : ${pred:.2f}" if pred else "Prediction    : None (parse failure)")
print(f"Ground truth  : ${truth:.2f}")
if pred:
    print(f"Error         : ${abs(pred - truth):.2f}")


Group Beam Search was moved to a `custom_generate` repo: https://hf.co/transformers-community/group-beam-search. To prevent loss of backward compatibility, add `custom_generate='transformers-community/group-beam-search'` to your `generate` call before v4.62.0.


generate.py: 0.00B [00:00, ?B/s]

beam_search.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/generate.py
- beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Passing `generation_config` together with generation-related arguments=({'max_length', 'diversity_penalty', 'max_new_tokens', 'num_beam_groups', 'num_beams', 'pad_token_id', 'num_return_sequences', 'use_cache', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not 

Prompt (tail) : ...itching, soft‑touch bypass, and expression jack for dynamic control.

Price is $
Prediction    : $2.00
Ground truth  : $219.00
Error         : $217.00


In [17]:
# set_seed(42)
# base_tester = evaluate(base_model_predict, test, size=EVAL_SIZE)


### 8b · Base Model — Full Dataset Test Set

Load `items_prompts_full` test split and evaluate the base model on it for comparison.

In [18]:
# Load full-dataset test split (inference only — not used for training)
full_test = load_dataset("ed-donner/items_prompts_full")["test"]
print(f"Full test : {len(full_test):,} rows")


README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/172M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/2.15M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Full test : 10,000 rows


In [19]:
# set_seed(42)
# base_tester_full = evaluate(base_model_predict, full_test, size=len(full_test))


## 9 · Prepare Dataset — Harmony Format for SFT

For full fine-tuning, `SFTTrainer` needs a single `text` field containing the **complete sequence** (user turn + assistant turn + EOS). We use `apply_chat_template` with `include_completion=True` to build properly harmony-formatted training sequences.

**Why not raw concatenation?** GPT-OSS will ignore price tokens that appear outside its expected harmony structure — training on bare `prompt + completion` strings would teach the model nothing.


In [20]:
def format_for_sft(examples):
    """
    Build full harmony-formatted training sequences.
    Each sequence: [harmony user turn][harmony assistant turn with price][EOS]
    """
    texts = []
    for prompt, completion in zip(examples["prompt"], examples["completion"]):
        item = {"prompt": prompt, "completion": completion}
        full_text = build_harmony_prompt(item, include_completion=True)
        # Append EOS so the model learns a clean stop after the price
        if not full_text.endswith(tokenizer.eos_token):
            full_text += tokenizer.eos_token
        texts.append(full_text)
    return {"text": texts}


train_sft = train.map(format_for_sft, batched=True, remove_columns=train.column_names)
val_sft   = val.map(format_for_sft,   batched=True, remove_columns=val.column_names)

# Verify token length distribution — ensure MAX_SEQ_LENGTH covers the data
print("Sample SFT text:")
print(train_sft[0]["text"][:400])
sample_lengths = [len(tokenizer.encode(t)) for t in train_sft.select(range(200))["text"]]
print(f"\nToken length (200 samples) — min: {min(sample_lengths)}  max: {max(sample_lengths)}  mean: {sum(sample_lengths)/len(sample_lengths):.0f}")
print(f"MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} covers {sum(l<=MAX_SEQ_LENGTH for l in sample_lengths)/len(sample_lengths)*100:.1f}% of samples")
print(f"\nTrain SFT size : {len(train_sft):,}")
print(f"Val SFT size   : {len(val_sft):,}")


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Sample SFT text:
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-03-08

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>What does this cost to the nearest dollar?

Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  
Category: 

Token length (200 samples) — min: 134  max: 198  mean: 171
MAX_SEQ_LENGTH=512 covers 100.0% of samples

Train SFT size : 20,000
Val SFT size   : 1,000


## 10 · Full Fine-Tuning with DeepSpeed ZeRO-3

### Memory budget on a single H100 80 GB

| Component | Bytes | Notes |
|---|---|---|
| MXFP4 model weights | ~64 GB | 117B × 4.25 bits ÷ 8 |
| Active expert gradients (per step) | ~5 GB bf16 | only activated experts per token |
| Activations (gradient checkpointing) | ~4 GB | recomputed, not stored |
| KV cache (batch=1, seq=512) | ~1 GB | |
| **Total on-GPU** | **~74 GB** | ≤ 80 GB ✓ |
| Optimizer states (Adafactor) | ~234 GB bf16 | **offloaded to CPU RAM** |

**ZeRO Stage 3 + CPU offload handles everything beyond the ~74 GB on-GPU budget.**  
Ensure the H100 node has ≥ 256 GB host RAM for Adafactor CPU states.

### Why Adafactor over AdamW?
- AdamW stores 2 copies of every parameter (first + second moment) → ~468 GB for 120B model
- Adafactor uses factored second-moment estimates → ~234 GB — half the optimizer memory
- No first moment: slightly noisier but stable with the cosine LR + warmup schedule used here


In [21]:
# ── Free base model VRAM before training run ────────────────────────────────
# base_model was loaded for evaluation. ZeRO-3 will reload and shard the weights
# automatically — we must free the eval copy first to avoid OOM.
del base_model
torch.cuda.empty_cache()
print(f"VRAM after cleanup : {torch.cuda.memory_allocated()/1e9:.1f} GB")


VRAM after cleanup : 10.2 GB


In [27]:
# SFTConfig replaces TrainingArguments in TRL ≥ 0.9 and accepts dataset_text_field / packing directly.
# We pass deepspeed=DS_CONFIG_PATH to activate ZeRO-3 sharding.

training_args = SFTConfig(
    # ── Output ─────────────────────────────────────────────────────────────
    output_dir              = CHECKPOINT_DIR,

    # ── Training duration ──────────────────────────────────────────────────
    num_train_epochs        = EPOCHS,
    max_steps               = -1,

    # ── Batch & gradient ───────────────────────────────────────────────────
    # batch=1: mandatory for 120B MoE on single H100 even with gradient checkpointing
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = GRAD_ACCUM,
    gradient_checkpointing      = True,        # recompute activations — essential for 120B
    gradient_checkpointing_kwargs = {"use_reentrant": False},

    # ── Optimiser & LR ─────────────────────────────────────────────────────
    optim               = OPTIMIZER,           # adafactor
    learning_rate       = LEARNING_RATE,
    weight_decay        = WEIGHT_DECAY,
    warmup_steps        = WARMUP_STEPS,
    lr_scheduler_type   = LR_SCHEDULER,
    max_grad_norm       = MAX_GRAD_NORM,
    # Adafactor note: scale_parameter and relative_step must be False when
    # using an explicit LR schedule — otherwise Adafactor ignores learning_rate
    # adafactor_scale_parameter = False,
    # adafactor_relative_step   = False,
    # adafactor_warmup_init     = False,

    # ── Precision (H100) ───────────────────────────────────────────────────
    fp16                = False,
    bf16                = USE_BF16,

    # ── DeepSpeed ──────────────────────────────────────────────────────────
    # deepspeed           = DS_CONFIG_PATH,      # ZeRO-3 + CPU offload

    # ── Sequence / packing ─────────────────────────────────────────────────
    max_length      = MAX_SEQ_LENGTH,
    packing             = True,                # pack short prompts into full windows
    dataset_text_field  = "text",

    # ── Logging ────────────────────────────────────────────────────────────
    logging_steps       = LOG_STEPS,
    report_to           = "wandb" if LOG_TO_WANDB else "none",
    run_name            = RUN_NAME,

    # ── Checkpointing ──────────────────────────────────────────────────────
    save_strategy       = "steps",
    save_steps          = SAVE_STEPS,
    save_total_limit    = 3,

    # ── Evaluation ─────────────────────────────────────────────────────────
    eval_strategy       = "steps",
    eval_steps          = SAVE_STEPS,

    # ── Hub push ───────────────────────────────────────────────────────────
    # push_to_hub disabled: each checkpoint is ~63 GB (full MXFP4 model).
    # Do a single explicit push after training completes (Section 11).
    push_to_hub         = False,
    # hub_model_id      = HUB_MODEL_NAME,
    # hub_strategy      = "every_save",
    # hub_private_repo  = True,

    # ── Misc ───────────────────────────────────────────────────────────────
    seed                    = 3407,
    dataloader_num_workers  = 4,
    save_only_model         = True,    # skip optimizer states in checkpoint (~234 GB/ckpt saved)
    remove_unused_columns   = False,
)

print(f"Output dir     : {CHECKPOINT_DIR}")
print(f"Hub target     : {HUB_MODEL_NAME}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"DeepSpeed cfg  : {DS_CONFIG_PATH}")


Output dir     : /tmp/checkpoint_dir/price-gpt-oss-20b-2026-03-08_01.22.31-lite
Hub target     : martinsawojide/price-gpt-oss-20b-2026-03-08_01.22.31-lite
Effective batch: 8
DeepSpeed cfg  : /tmp/mawojide/ds_zero3.json


In [24]:
# del base_model
# torch.cuda.empty_cache()
# gc.collect()
# print(f"VRAM after cleanup : {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [28]:
# SFTTrainer loads the model internally, sharded by ZeRO-3.
# Do NOT load the model yourself here — let the trainer manage sharding.
# If you pass an already-loaded model, ZeRO-3 cannot shard it correctly.

# trainer = SFTTrainer(
#     model            = BASE_MODEL,   # string — trainer loads fresh in bf16 via ZeRO-3
#     processing_class = tokenizer,
#     train_dataset    = train_sft,
#     eval_dataset     = val_sft,
#     args             = training_args,
# )

from peft import LoraConfig

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

trainer = SFTTrainer(
    model            = BASE_MODEL,
    processing_class = tokenizer,
    train_dataset    = train_sft,
    eval_dataset     = val_sft,
    args             = training_args,
    peft_config      = lora_config,
)

total_params = sum(p.numel() for p in trainer.model.parameters())
trainable    = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable:,}  ({100 * trainable / total_params:.1f}% — full fine-tune)")


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-

ValueError: The model you are trying to fine-tune is quantized with QuantizationMethod.MXFP4 but that quantization method do not support training. Please open an issue on GitHub: https://github.com/huggingface/transformers to request the support for training support for QuantizationMethod.MXFP4

### Start training

Checkpoints are pushed to HuggingFace Hub every `SAVE_STEPS` steps.  
If the session is interrupted, resume from the latest checkpoint:
```python
trainer.train(resume_from_checkpoint=True)
```


In [ ]:
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total steps    : {trainer_stats.global_step}")
print(f"Training loss  : {trainer_stats.training_loss:.4f}")
print(f"Runtime        : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")


## 11 · Save Fine-Tuned Model

For full fine-tuning there are no separate LoRA adapter weights — we save the entire model. `save_pretrained` with ZeRO-3 consolidates the sharded weights back into a single checkpoint before writing.


In [ ]:
# ZeRO-3 requires consolidating shards before save
# If using deepspeed CLI: deepspeed --zero_to_fp32 to reconstruct weights
# With trainer.save_model(), TRL handles shard consolidation automatically
trainer.save_model(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Saved locally  → {CHECKPOINT_DIR}/")

# Push full model to Hub (not just adapter)
trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Pushed to Hub  → {HUB_MODEL_NAME}")


## 12 · Load Fine-Tuned Model for Inference

Free the training model from GPU first — inference evaluation does not need ZeRO-3 sharding.


In [ ]:
# ── Free training model ─────────────────────────────────────────────────────
del trainer
torch.cuda.empty_cache()
print(f"VRAM after training cleanup : {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
# ── If loading from a previous saved run instead of the one just trained ────
# HUB_MODEL_NAME = "martinsawojide/price-gpt-oss-120b-YYYY-MM-DD_HH.MM.SS"
# or use the local checkpoint:
# LOAD_FROM = CHECKPOINT_DIR

LOAD_FROM = HUB_MODEL_NAME   # change to CHECKPOINT_DIR to load locally

ft_tokenizer = AutoTokenizer.from_pretrained(LOAD_FROM)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"

fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    LOAD_FROM,
    torch_dtype  = torch.bfloat16,
    device_map   = "cuda",
    attn_implementation = "flash_attention_2",
)
fine_tuned_model.eval()
fine_tuned_model.generation_config.reasoning_effort = REASONING_EFFORT

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"Fine-tuned model loaded : {LOAD_FROM}")
print(f"VRAM used               : {vram_used:.1f} GB")


## 13 · Fine-Tuned Model Evaluation

Same inference recipe as base model evaluation for a fair apples-to-apples comparison.


In [ ]:
def fine_tuned_predict(item):
    """Inference wrapper using the fine-tuned model — same beam config as base eval."""
    return predict_with_model(fine_tuned_model, ft_tokenizer, item)


In [ ]:
# Smoke test — compare single item before committing to full eval
sample    = test[0]
pred_ft   = fine_tuned_predict(sample)
truth     = float(sample["completion"])

print(f"Prompt tail    : ...{sample['prompt'][-80:]}")
print(f"FT prediction  : ${pred_ft:.2f}" if pred_ft else "FT prediction  : None")
print(f"Ground truth   : ${truth:.2f}")
if pred_ft:
    print(f"Error          : ${abs(pred_ft - truth):.2f}")


In [ ]:
set_seed(42)
ft_tester = evaluate(fine_tuned_predict, test, size=EVAL_SIZE)


### 13b · Fine-Tuned Model — Full Dataset Test Set

In [ ]:
# set_seed(42)
# ft_tester_full = evaluate(fine_tuned_predict, full_test, size=len(full_test))


## 14 · Side-by-Side Comparison & Results

In [ ]:
# def tester_metrics(tester):
#     n   = len(tester.errors)
#     mae = sum(tester.errors) / n
#     mse = mean_squared_error(tester.truths, tester.guesses)
#     r2  = r2_score(tester.truths, tester.guesses) * 100
#     return mae, mse, r2

# bm_lite, bm_lite_mse, bm_lite_r2 = tester_metrics(base_tester)
# bm_full, bm_full_mse, bm_full_r2 = tester_metrics(base_tester_full)
# ft_lite, ft_lite_mse, ft_lite_r2 = tester_metrics(ft_tester)
# ft_full, ft_full_mse, ft_full_r2 = tester_metrics(ft_tester_full)

# print(f"{'Metric':<10} {'Base·Lite':>14} {'FT·Lite':>14} {'Base·Full':>14} {'FT·Full':>14}")
# print("─" * 68)
# print(f"{'MAE ($)':<10} {bm_lite:>14.2f} {ft_lite:>14.2f} {bm_full:>14.2f} {ft_full:>14.2f}")
# print(f"{'MSE':<10} {bm_lite_mse:>14.0f} {ft_lite_mse:>14.0f} {bm_full_mse:>14.0f} {ft_full_mse:>14.0f}")
# print(f"{'r² (%)':<10} {bm_lite_r2:>14.1f} {ft_lite_r2:>14.1f} {bm_full_r2:>14.1f} {ft_full_r2:>14.1f}")
# print("─" * 68)
# print(f"\nLite  — fine-tuning improved MAE by ${bm_lite - ft_lite:.2f} ({(bm_lite - ft_lite)/bm_lite*100:.1f}%)")
# print(f"Full  — fine-tuning improved MAE by ${bm_full - ft_full:.2f} ({(bm_full - ft_full)/bm_full*100:.1f}%)")


In [ ]:
# # Overlay scatter — 4 traces: base & FT × lite & full test sets
# fig = go.Figure()

# for tester, name, color in [
#     (base_tester,      "Base GPT-OSS 120B · Lite test",       "royalblue"),
#     (ft_tester,        "Fine-Tuned GPT-OSS 120B · Lite test",  "crimson"),
#     (base_tester_full, "Base GPT-OSS 120B · Full test",        "cornflowerblue"),
#     (ft_tester_full,   "Fine-Tuned GPT-OSS 120B · Full test",  "darkorange"),
# ]:
#     fig.add_trace(go.Scatter(
#         x=tester.truths, y=tester.guesses,
#         mode="markers", name=name,
#         marker=dict(size=5, opacity=0.5),
#     ))

# max_val = max(max(t.truths + t.guesses) for t in [base_tester, ft_tester, base_tester_full, ft_tester_full])
# fig.add_trace(go.Scatter(
#     x=[0, max_val], y=[0, max_val], mode="lines",
#     line=dict(dash="dash", color="gray", width=1),
#     name="Perfect prediction", hoverinfo="skip",
# ))

# fig.update_layout(
#     title=(
#         f"Base vs Fine-Tuned GPT-OSS 120B — Lite & Full test sets<br>"
#         f"<sup>Lite MAE: Base ${bm_lite:.2f} → FT ${ft_lite:.2f}  |  "
#         f"Full MAE: Base ${bm_full:.2f} → FT ${ft_full:.2f}</sup>"
#     ),
#     xaxis_title="Actual Price ($)", yaxis_title="Predicted Price ($)",
#     width=960, height=640, template="plotly_white",
# )
# fig.update_xaxes(range=[0, max_val])
# fig.update_yaxes(range=[0, max_val])
# fig.show()
